# GoEmotions Multi-Label Classification with RoBERTa

Clean Trainer-based implementation for easy hyperparameter tuning.

Task: Multi-label emotion classification.


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset


## Load Dataset

In [ ]:
# Replace with your dataset path
df = pd.read_csv('goemotions.csv')

# Keep only text + emotion columns
emotion_cols = [c for c in df.columns if c != 'text']
df = df[['text'] + emotion_cols]

print('Number of emotions:', len(emotion_cols))


## Train / Validation / Test Split

In [ ]:
df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42)

train_texts = df_train['text'].tolist()
val_texts = df_val['text'].tolist()
test_texts = df_test['text'].tolist()

y_train = df_train[emotion_cols].values
y_val = df_val[emotion_cols].values
y_test = df_test[emotion_cols].values


## Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)


## Dataset Class (Trainer Compatible)

In [ ]:
class GoEmotionsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = GoEmotionsDataset(train_encodings, y_train)
val_dataset = GoEmotionsDataset(val_encodings, y_val)
test_dataset = GoEmotionsDataset(test_encodings, y_test)


## Load Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=len(emotion_cols)
)

model.config.problem_type = 'multi_label_classification'


## Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0)
    }


## Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir='./goemotions_roberta',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    logging_steps=100,
    warmup_steps=200,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)


## Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


## Final Evaluation on Test Set

In [ ]:
trainer.evaluate(test_dataset)
